This notebook processes the experimentally validated TSS sites. The raw table are the selected columns from table S2 of Chen *et al.* 2013. The deliverables of this notebook:

- Clean the raw table `data/raw/baculovirus/Baculovirus_TSS.csv`
- Extract positive and negative sets for TSS prediction task.
- Generate local datasets loadable as `datasets.DatasetDict` following [Hugging Face document](https://huggingface.co/docs/datasets/create_dataset). The processed datasets is the baculovirus counterpart of dataset for promoter existance prediction task in [`"InstaDeepAI/nucleotide_transformer_downstream_tasks_revised"`](https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised). The the datasets structures should be identical.

# Clean TSS Table

In [1]:
import pandas as pd

In [2]:
raw_df = pd.read_csv("../data/raw/baculovirus/Baculovirus_TSS.csv").dropna(axis=0, subset=['TSS'])

In [3]:
raw_df.shape

(216, 9)

Different from the authors' claim of 220 TSS, there are 4 sites not recorded in column "All TSS sites" in the original table. Since the original table was clearly manually curated, I decided not to challenge the reason of their removal and accept the authors' decision.

In [4]:
ffill_cols = ["Direction", "ORF", "ORF start", "ORF end"]
raw_df[ffill_cols] = raw_df[ffill_cols].ffill()

In [5]:
raw_df.columns

Index(['TSS', 'Direction', 'TATAA_Distance_from_TSS', 'CAGT_Distance_from_TSS',
       'TAAG_Distance_from_TSS', 'ORF', 'ORF start', 'ORF end',
       'Early / Late'],
      dtype='str')

In [6]:
intcols = [
    'TSS', 
    'ORF start', 
    'ORF end'
]
raw_df[intcols] = raw_df[intcols].astype('int', errors='ignore')

In [7]:
raw_df.describe()

,TSS,TATAA_Distance_from_TSS,ORF start,ORF end
count,216.00000,75.000000,216.000000,216.000000
mean,66456.99537,-11.706667,65946.421296,66814.504630
std,38286.42905,23.959861,38290.847026,38332.276286
min,482.00000,-47.000000,503.000000,1009.000000
25%,30923.25000,-30.000000,30288.750000,30860.750000
50%,64537.00000,-15.000000,63678.000000,64374.500000
75%,99781.00000,-3.000000,99830.500000,100029.750000
max,133588.00000,44.000000,133591.000000,133836.000000


For comma seperated "Distance_from_TSS" columns, choose the most distant value.

In [8]:
def pick_distance(s):
    if s:
        if isinstance(s, str):
            vals = [int(v) for v in s.split(",")]
            return min(vals)
        else:
            return s

In [9]:
distance_cols = [
    'TATAA_Distance_from_TSS', 
    'CAGT_Distance_from_TSS', 
    'TAAG_Distance_from_TSS', 
]
raw_df[distance_cols] = raw_df[distance_cols].map(pick_distance)

In [10]:
raw_df.TATAA_Distance_from_TSS.unique()

array([ nan,  43.,  -3., -30.,  -9., -29., -28., -32.,  33., -24., -15.,
       -25., -46., -44., -40.,  24.,  21., -27.,  44.,  -7.,  22.,   3.,
       -18., -33.,  23., -35.,  37., -26., -47.,  13., -31.,   6.])

In [11]:
raw_df.isna().sum()

TSS                          0
Direction                    0
TATAA_Distance_from_TSS    141
CAGT_Distance_from_TSS     151
TAAG_Distance_from_TSS      75
ORF                          0
ORF start                    0
ORF end                      0
Early / Late                 0
dtype: int64

In [12]:
raw_df.dtypes

TSS                          int64
Direction                      str
TATAA_Distance_from_TSS    float64
CAGT_Distance_from_TSS     float64
TAAG_Distance_from_TSS     float64
ORF                            str
ORF start                    int64
ORF end                      int64
Early / Late                   str
dtype: object

In [13]:
raw_df.to_csv('../data/interim/tss.csv', index=False)

# Extract Positive and Negative Sets

In [14]:
from Bio import SeqIO

In [22]:
with open("../data/raw/baculovirus/Baculoviridae/NC_001623.1.gb", 'r') as f:
    seq = next(iter(SeqIO.parse(f, format='gb')))

In [23]:
seq

SeqRecord(seq=Seq('GAATTCTACCCGTAAAGCGAGTTTAGTTTTGAAAAACAAATGACATCATTTGTA...GTA'), id='NC_001623.1', name='NC_001623', description='Autographa californica nucleopolyhedrovirus, complete genome', dbxrefs=['BioProject:PRJNA485481'])

In [31]:
len(seq)

133894

The sequence loaded by Biopython is 0-based. 

### TSS Table

The positions in TSS table are 1-based. Also note that the ORFs in TSS table are 1-based, double closed regions.

The motif distance follows the strand, with positive numbers indicating downstream and negative numbers indicating upstream. For TSS on positive strand, the motif starts at *TSS + Motif_distance* (1-based, included). For TSS on negative strand, the motif starts at *TSS - Motif_distance* (1-based, included)

For a TSS on positive strand, the motif distance -1 means the motif starts at 1 bp upstream. For a TSS on negative strand, the motif distance 10 means the motif starts at 10 bp downstream. In both cases the motif indices are smaller than TSS, i.e. on the left.

Note that the **distances in TSS table are not always precise**, maybe because of genome version updates, ORC recognition, or errors introduced in manual curation.

Take row 11 (1-based) as an example. TSS: 6885, TAAG motif distance: 4, ORF: 6917-7735. The true TAAG motif distance should be -1.

In [66]:
seq[6883:6890]

SeqRecord(seq=Seq('TAAGATT'), id='NC_001623.1', name='NC_001623', description='Autographa californica nucleopolyhedrovirus, complete genome', dbxrefs=[])

In [50]:
# pos strand
tss = 79902
motif_dist = -48
st = tss+motif_dist-1 
seq[st:st+5]

SeqRecord(seq=Seq('CAGTA'), id='NC_001623.1', name='NC_001623', description='Autographa californica nucleopolyhedrovirus, complete genome', dbxrefs=[])

In [65]:
# neg strand
tss = 2272
motif_dist = -1
st = tss-motif_dist 
seq[st-5:st].reverse_complement()

SeqRecord(seq=Seq('TAAGA'), id='<unknown id>', name='<unknown name>', description='<unknown description>', dbxrefs=[])

### Strategy

In accordance with the human promoter existance prediction task, we choose 300bp as the sequence length.

The 216 experimentally verified TSS sites are our positive set.

For positive TSS record:

- Do not include sequences in ORF (the coding region would be too obvious).
- Include all existing motifs.

For negative TSS record:

- Use intergenic regions (CDS region can be loaded from `seq.features` but UTR regions are not included). For this task, I'm 
- Keep distance from TSS regions on both strands.